# CSL7110 Assignment 3 - Recommender Systems

MovieLens Small dataset. All parts implemented from scratch using numpy, pandas and sklearn.

## Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from scipy.sparse.linalg import svds

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')


## Load Data

In [ ]:
movies  = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')
tags    = pd.read_csv('ml-latest-small/tags.csv')

print('movies :', movies.shape)
print('ratings:', ratings.shape)
print('tags   :', tags.shape)
movies.head()


In [ ]:
ratings.head()


In [ ]:
print('users  :', ratings['userId'].nunique())
print('movies :', ratings['movieId'].nunique())
print('rating range:', ratings['rating'].min(), '-', ratings['rating'].max())
print()
print(movies['genres'].value_counts().head(10))


---
## Part 1 - Content-Based Filtering

### Task 1: TF-IDF Based Recommendation

In [ ]:
# replace | with space so each genre becomes a separate token
movies['genre_str'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['genre_str'] = movies['genre_str'].replace('(no genres listed)', '')

movies[['title', 'genre_str']].head(10)


In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genre_str'])

print('tfidf matrix shape:', tfidf_matrix.shape)
print('genres found:', tfidf.get_feature_names_out())


In [ ]:
# linear_kernel is faster than cosine_similarity for normalised tfidf vectors
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print('similarity matrix shape:', cosine_sim.shape)

# map title -> row position
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


In [ ]:
def get_tfidf_recommendations(title, top_n=5):
    if title not in indices:
        return f"'{title}' not found"
    
    idx = indices[title]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:top_n+1]  # skip itself
    
    movie_indices = [s[0] for s in scores]
    sim_values    = [s[1] for s in scores]
    
    result = movies['title'].iloc[movie_indices].reset_index(drop=True)
    return pd.DataFrame({'Recommended Movie': result, 'Cosine Similarity': sim_values})


In [ ]:
test_titles = ['Toy Story (1995)', 'Jumanji (1995)', 'Goodfellas (1990)']

for title in test_titles:
    print('\nInput:', title)
    print(get_tfidf_recommendations(title).to_string(index=False))


In [ ]:
# heatmap showing genre similarity for the 15 most-rated movies
top_ids = ratings.groupby('movieId')['rating'].count().nlargest(15).index
top_movies_subset = movies[movies['movieId'].isin(top_ids)].copy()

tfidf2 = TfidfVectorizer(stop_words='english')
mat2   = tfidf2.fit_transform(top_movies_subset['genre_str'])
sim2   = linear_kernel(mat2, mat2)

labels = [t[:20] for t in top_movies_subset['title'].tolist()]

plt.figure(figsize=(12, 9))
sns.heatmap(sim2, xticklabels=labels, yticklabels=labels,
            cmap='YlOrRd', annot=False, linewidths=0.5)
plt.title('Genre Cosine Similarity - Top 15 Most Rated Movies')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('task1_heatmap.png', dpi=100)
plt.show()


### Task 2: User-Profile-Based Content Recommender

For each user we build a profile vector by taking a weighted average of the TF-IDF vectors of movies they rated, using the ratings as weights.

In [ ]:
# map movieId -> row index in tfidf_matrix
movie_idx_map = pd.Series(movies.index, index=movies['movieId']).drop_duplicates()

def build_user_profiles(ratings_df, tfidf_mat, movie_idx_map):
    profiles = {}
    for user_id, group in ratings_df.groupby('userId'):
        group = group[group['movieId'].isin(movie_idx_map.index)]
        if group.empty:
            continue
        row_idxs     = movie_idx_map[group['movieId']].values
        user_ratings = group['rating'].values
        
        # weighted sum then normalise by total rating mass
        weighted = tfidf_mat[row_idxs].multiply(user_ratings[:, None])
        profile  = weighted.sum(axis=0) / user_ratings.sum()
        profiles[user_id] = np.asarray(profile).flatten()
    return profiles

user_profiles = build_user_profiles(ratings, tfidf_matrix, movie_idx_map)
print(f'profiles built: {len(user_profiles)} users')
print(f'profile dimension: {list(user_profiles.values())[0].shape}')


In [ ]:
def get_user_recommendations(user_id, user_profiles, tfidf_mat, movies_df, ratings_df, top_n=10):
    if user_id not in user_profiles:
        return 'user not found'
    
    profile    = user_profiles[user_id].reshape(1, -1)
    sim_scores = cosine_similarity(profile, tfidf_mat).flatten()
    
    # mask out movies the user already rated
    rated_ids      = ratings_df[ratings_df['userId'] == user_id]['movieId'].tolist()
    rated_positions = movie_idx_map[movie_idx_map.index.isin(rated_ids)].values
    sim_scores[rated_positions] = -1
    
    top_idx = np.argsort(sim_scores)[::-1][:top_n]
    result  = movies_df.iloc[top_idx][['title', 'genres']].copy()
    result['Similarity'] = sim_scores[top_idx]
    return result.reset_index(drop=True)

print('Top 10 recommendations for user 1:')
print(get_user_recommendations(1, user_profiles, tfidf_matrix, movies, ratings).to_string(index=False))


In [ ]:
def precision_recall_at_k(user_id, user_profiles, tfidf_mat, movies_df,
                           ratings_df, movie_idx_map, k=10, threshold=4.0):
    relevant = ratings_df[
        (ratings_df['userId'] == user_id) & (ratings_df['rating'] >= threshold)
    ]['movieId'].tolist()
    
    if not relevant or user_id not in user_profiles:
        return None, None
    
    profile    = user_profiles[user_id].reshape(1, -1)
    sim_scores = cosine_similarity(profile, tfidf_mat).flatten()
    
    top_idx       = np.argsort(sim_scores)[::-1][:k]
    recommended   = movies_df.iloc[top_idx]['movieId'].tolist()
    
    hits      = len(set(recommended) & set(relevant))
    precision = hits / k
    recall    = hits / len(relevant)
    return precision, recall

precisions, recalls = [], []
for uid in list(user_profiles.keys())[:200]:
    p, r = precision_recall_at_k(uid, user_profiles, tfidf_matrix, movies,
                                  ratings, movie_idx_map, k=10)
    if p is not None:
        precisions.append(p)
        recalls.append(r)

print(f'Precision@10: {np.mean(precisions):.4f}')
print(f'Recall@10   : {np.mean(recalls):.4f}')


---
## Part 2 - Collaborative Filtering

### Task 3: User-Based Collaborative Filtering

In [ ]:
user_movie_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')
print('shape  :', user_movie_matrix.shape)
print(f'missing: {user_movie_matrix.isna().sum().sum() / user_movie_matrix.size * 100:.1f}%')


In [ ]:
def pearson_user_similarity(matrix):
    # mean-centre each row, then cosine similarity on centred vectors = pearson
    mat      = matrix.values.astype(float)
    means    = np.nanmean(mat, axis=1, keepdims=True)
    centred  = np.nan_to_num(mat - means)
    norms    = np.linalg.norm(centred, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10
    normed   = centred / norms
    sim      = normed @ normed.T
    return pd.DataFrame(sim, index=matrix.index, columns=matrix.index)

print('computing user-user similarity...')
user_sim_df = pearson_user_similarity(user_movie_matrix)
print('done. shape:', user_sim_df.shape)


In [ ]:
def predict_rating_user_cf(user_id, movie_id, matrix, sim_df, k=20):
    if movie_id not in matrix.columns:
        return np.nan
    
    movie_ratings = matrix[movie_id].dropna()
    neighbors     = movie_ratings.index.difference([user_id])
    if neighbors.empty:
        return np.nan
    
    sims  = sim_df.loc[user_id, neighbors]
    top_k = sims.nlargest(k)
    
    denom = top_k.abs().sum()
    if denom == 0:
        return np.nan
    
    user_mean      = matrix.loc[user_id].mean(skipna=True)
    neighbor_means = matrix.loc[top_k.index].mean(axis=1, skipna=True)
    numerator      = (top_k * (movie_ratings.loc[top_k.index] - neighbor_means)).sum()
    
    return user_mean + numerator / denom

# quick check
pred   = predict_rating_user_cf(1, 318, user_movie_matrix, user_sim_df)
actual = user_movie_matrix.loc[1, 318]
print(f'user 1, movie 318 | actual: {actual} | predicted: {pred:.3f}')


In [ ]:
def recommend_user_cf(user_id, matrix, sim_df, movies_df, top_n=10, k=20):
    rated   = matrix.loc[user_id].dropna().index.tolist()
    unrated = [m for m in matrix.columns if m not in rated]
    
    preds = {}
    for mid in unrated:
        p = predict_rating_user_cf(user_id, mid, matrix, sim_df, k)
        if not np.isnan(p):
            preds[mid] = p
    
    top_movies = sorted(preds, key=preds.get, reverse=True)[:top_n]
    result     = movies_df[movies_df['movieId'].isin(top_movies)][['title', 'genres']].copy()
    result['Predicted Rating'] = result['movieId'].map(preds)
    result = result.sort_values('Predicted Rating', ascending=False)
    return result.reset_index(drop=True)

print('User-CF top 10 for user 1 (k=20):')
recs_ucf = recommend_user_cf(1, user_movie_matrix, user_sim_df, movies)
print(recs_ucf.to_string(index=False))


In [ ]:
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

train_matrix   = train_ratings.pivot(index='userId', columns='movieId', values='rating')
train_user_sim = pearson_user_similarity(train_matrix)

test_sample = test_ratings.sample(min(500, len(test_ratings)), random_state=42)

actuals, preds_list = [], []
for _, row in test_sample.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if uid in train_matrix.index:
        p = predict_rating_user_cf(uid, mid, train_matrix, train_user_sim, k=20)
        if not np.isnan(p):
            actuals.append(row['rating'])
            preds_list.append(np.clip(p, 0.5, 5.0))

rmse_ucf = np.sqrt(mean_squared_error(actuals, preds_list))
print(f'User-CF RMSE: {rmse_ucf:.4f}  ({len(actuals)} predictions)')


In [ ]:
def cf_precision_recall_at_k(user_id, train_matrix, test_ratings, sim_df,
                               k=10, threshold=4.0):
    gt = test_ratings[
        (test_ratings['userId'] == user_id) & (test_ratings['rating'] >= threshold)
    ]['movieId'].tolist()
    
    if not gt or user_id not in train_matrix.index:
        return None, None
    
    rated   = train_matrix.loc[user_id].dropna().index.tolist()
    unrated = [m for m in train_matrix.columns if m not in rated]
    
    preds = {}
    for mid in unrated[:300]:
        p = predict_rating_user_cf(user_id, mid, train_matrix, sim_df, k=15)
        if not np.isnan(p):
            preds[mid] = p
    
    top_k = sorted(preds, key=preds.get, reverse=True)[:k]
    hits  = len(set(top_k) & set(gt))
    return hits / k, hits / len(gt)

p_list, r_list = [], []
for uid in test_ratings['userId'].unique()[:30]:
    p, r = cf_precision_recall_at_k(uid, train_matrix, test_ratings, train_user_sim)
    if p is not None:
        p_list.append(p)
        r_list.append(r)

print(f'User-CF  Precision@10: {np.mean(p_list):.4f}  Recall@10: {np.mean(r_list):.4f}')


In [ ]:
k_values  = [5, 10, 20, 30, 50]
rmse_by_k = []

for k in k_values:
    a_tmp, p_tmp = [], []
    for _, row in test_sample.iterrows():
        uid, mid = int(row['userId']), int(row['movieId'])
        if uid in train_matrix.index:
            p = predict_rating_user_cf(uid, mid, train_matrix, train_user_sim, k=k)
            if not np.isnan(p):
                a_tmp.append(row['rating'])
                p_tmp.append(np.clip(p, 0.5, 5.0))
    rmse_by_k.append(np.sqrt(mean_squared_error(a_tmp, p_tmp)) if a_tmp else np.nan)

plt.figure()
plt.plot(k_values, rmse_by_k, 'o-', color='steelblue', linewidth=2)
plt.xlabel('K (number of neighbours)')
plt.ylabel('RMSE')
plt.title('User-CF RMSE vs K')
plt.xticks(k_values)
plt.tight_layout()
plt.savefig('task3_rmse_vs_k.png', dpi=100)
plt.show()

for k, r in zip(k_values, rmse_by_k):
    print(f'K={k:2d}  RMSE={r:.4f}')


### Task 4: Item-Based Collaborative Filtering

In [ ]:
print('computing item-item similarity...')

item_matrix = user_movie_matrix.T.fillna(0)
item_mat    = item_matrix.values.astype(float)

# mean-centre only over rated (non-zero) entries
row_means   = item_mat.mean(axis=1, keepdims=True)
mask        = item_mat != 0
centred     = np.where(mask, item_mat - row_means, 0)

norms_i = np.linalg.norm(centred, axis=1, keepdims=True)
norms_i[norms_i == 0] = 1e-10
normed_i = centred / norms_i
item_sim = normed_i @ normed_i.T

item_ids       = item_matrix.index.tolist()
item_id_to_idx = {mid: i for i, mid in enumerate(item_ids)}

print('item-item matrix shape:', item_sim.shape)


In [ ]:
def predict_rating_item_cf(user_id, movie_id, matrix, item_sim, item_id_to_idx, k=20):
    if movie_id not in item_id_to_idx or user_id not in matrix.index:
        return np.nan
    
    target_idx  = item_id_to_idx[movie_id]
    user_rated  = matrix.loc[user_id].dropna()
    rated_ids   = user_rated.index.tolist()
    
    rated_positions = [item_id_to_idx[m] for m in rated_ids if m in item_id_to_idx]
    if not rated_positions:
        return np.nan
    
    sims      = item_sim[target_idx, rated_positions]
    top_k_n   = min(k, len(sims))
    top_idx   = np.argsort(np.abs(sims))[::-1][:top_k_n]
    top_sims  = sims[top_idx]
    top_rtgs  = user_rated.values[top_idx]
    
    denom = np.abs(top_sims).sum()
    if denom == 0:
        return np.nan
    return (top_sims * top_rtgs).sum() / denom

pred_item = predict_rating_item_cf(1, 318, user_movie_matrix, item_sim, item_id_to_idx)
print(f'Item-CF prediction user 1, movie 318: {pred_item:.3f}')


In [ ]:
def recommend_item_cf(user_id, matrix, item_sim, item_id_to_idx, movies_df, top_n=10, k=20):
    rated   = matrix.loc[user_id].dropna().index.tolist()
    unrated = [m for m in matrix.columns if m not in rated]
    
    preds = {}
    for mid in unrated:
        p = predict_rating_item_cf(user_id, mid, matrix, item_sim, item_id_to_idx, k)
        if not np.isnan(p):
            preds[mid] = p
    
    top_movies = sorted(preds, key=preds.get, reverse=True)[:top_n]
    result     = movies_df[movies_df['movieId'].isin(top_movies)][['movieId', 'title', 'genres']].copy()
    result['Predicted Rating'] = result['movieId'].map(preds)
    return result.sort_values('Predicted Rating', ascending=False).reset_index(drop=True)

print('Item-CF top 10 for user 1:')
recs_icf = recommend_item_cf(1, user_movie_matrix, item_sim, item_id_to_idx, movies)
print(recs_icf[['title', 'genres', 'Predicted Rating']].to_string(index=False))


In [ ]:
a_item, p_item = [], []
for _, row in test_sample.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if uid in user_movie_matrix.index:
        p = predict_rating_item_cf(uid, mid, user_movie_matrix, item_sim, item_id_to_idx)
        if not np.isnan(p):
            a_item.append(row['rating'])
            p_item.append(np.clip(p, 0.5, 5.0))

rmse_icf = np.sqrt(mean_squared_error(a_item, p_item))
print(f'Item-CF RMSE: {rmse_icf:.4f}')
print(f'User-CF RMSE: {rmse_ucf:.4f}')

plt.figure(figsize=(6, 4))
plt.bar(['User-CF', 'Item-CF'], [rmse_ucf, rmse_icf], color=['steelblue', 'coral'])
plt.ylabel('RMSE')
plt.title('User-CF vs Item-CF')
plt.tight_layout()
plt.savefig('task4_cf_comparison.png', dpi=100)
plt.show()


In [ ]:
# Task 4 discussion question
print("""
Item-CF vs User-CF - efficiency in production systems
------------------------------------------------------
In real-world systems users number in the hundreds of millions while
the item catalogue grows slowly. Item-item similarities can be computed
offline once and cached. When a new user registers, we do not need to
recompute anything. User-CF, by contrast, requires keeping the full
user-user similarity matrix up to date as new users and ratings arrive.

On this dataset Item-CF also gives a lower RMSE, partly because the
item space is more stable and the similarity estimates are less noisy.

In general, for n_users >> n_items, Item-CF wins on both memory and speed.
""")


---
## Part 3 - Matrix Factorization

### Task 5: SVD

In [ ]:
# fill missing values with each user's mean before factorising
user_means = user_movie_matrix.mean(axis=1)
R_filled   = user_movie_matrix.T.fillna(user_means).T
R_matrix   = R_filled.values.astype(float)

user_ids_svd  = user_movie_matrix.index.tolist()
movie_ids_svd = user_movie_matrix.columns.tolist()

print('matrix shape:', R_matrix.shape)


In [ ]:
# mean-centre then decompose
global_mean = R_matrix.mean()
R_centred   = R_matrix - global_mean

k = 50
U, sigma, Vt = svds(R_centred, k=k)

print(f'U : {U.shape}')
print(f'sigma : {sigma.shape}')
print(f'Vt: {Vt.shape}')
print('top singular values:', np.sort(sigma)[::-1][:10].round(1))


In [ ]:
R_hat = np.clip(U @ np.diag(sigma) @ Vt + global_mean, 0.5, 5.0)
print('reconstructed matrix shape:', R_hat.shape)

def recommend_svd(user_id, R_hat, user_ids_svd, movie_ids_svd, movies_df, matrix, top_n=10):
    if user_id not in user_ids_svd:
        return 'user not found'
    
    u_idx    = user_ids_svd.index(user_id)
    pred_row = R_hat[u_idx, :].copy()
    
    # mask already-rated movies
    already_rated = ~matrix.loc[user_id].isna().values
    pred_row[already_rated] = -1
    
    top_idx     = np.argsort(pred_row)[::-1][:top_n]
    top_mids    = [movie_ids_svd[i] for i in top_idx]
    pred_dict   = {movie_ids_svd[i]: R_hat[u_idx, i] for i in top_idx}
    
    result = movies_df[movies_df['movieId'].isin(top_mids)][['movieId', 'title', 'genres']].copy()
    result['Predicted Rating'] = result['movieId'].map(pred_dict)
    return result.sort_values('Predicted Rating', ascending=False).reset_index(drop=True)

print('\nSVD top 10 for user 1:')
svd_recs = recommend_svd(1, R_hat, user_ids_svd, movie_ids_svd, movies, user_movie_matrix)
print(svd_recs[['title', 'genres', 'Predicted Rating']].to_string(index=False))


In [ ]:
a_svd, p_svd = [], []
for _, row in test_sample.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if uid in user_ids_svd and mid in movie_ids_svd:
        u_idx = user_ids_svd.index(uid)
        m_idx = movie_ids_svd.index(mid)
        a_svd.append(row['rating'])
        p_svd.append(R_hat[u_idx, m_idx])

rmse_svd = np.sqrt(mean_squared_error(a_svd, p_svd))
print(f'SVD RMSE (k={k}): {rmse_svd:.4f}')

def svd_precision_recall(user_id, R_hat, user_ids_svd, movie_ids_svd,
                          matrix, test_ratings, k=10, threshold=4.0):
    if user_id not in user_ids_svd:
        return None, None
    gt = test_ratings[
        (test_ratings['userId'] == user_id) & (test_ratings['rating'] >= threshold)
    ]['movieId'].tolist()
    if not gt:
        return None, None
    
    u_idx  = user_ids_svd.index(user_id)
    preds  = R_hat[u_idx, :].copy()
    rated  = ~matrix.loc[user_id].isna().values
    preds[rated] = -1
    
    top_k  = [movie_ids_svd[i] for i in np.argsort(preds)[::-1][:k]]
    hits   = len(set(top_k) & set(gt))
    return hits / k, hits / len(gt)

p_sv, r_sv = [], []
for uid in test_ratings['userId'].unique()[:50]:
    p, r = svd_precision_recall(uid, R_hat, user_ids_svd, movie_ids_svd,
                                 user_movie_matrix, test_ratings)
    if p is not None:
        p_sv.append(p); r_sv.append(r)

print(f'SVD Precision@10: {np.mean(p_sv):.4f}  Recall@10: {np.mean(r_sv):.4f}')


In [ ]:
k_list   = [10, 20, 50, 100, 150]
svd_rmse = []

for kk in k_list:
    kk = min(kk, min(R_centred.shape) - 1)
    U_k, s_k, Vt_k = svds(R_centred, k=kk)
    R_k = np.clip(U_k @ np.diag(s_k) @ Vt_k + global_mean, 0.5, 5.0)
    
    a_k, p_k = [], []
    for _, row in test_sample.iterrows():
        uid, mid = int(row['userId']), int(row['movieId'])
        if uid in user_ids_svd and mid in movie_ids_svd:
            a_k.append(row['rating'])
            p_k.append(R_k[user_ids_svd.index(uid), movie_ids_svd.index(mid)])
    svd_rmse.append(np.sqrt(mean_squared_error(a_k, p_k)))

plt.figure()
plt.plot(k_list, svd_rmse, 's-', color='purple', linewidth=2)
plt.xlabel('k (latent factors)')
plt.ylabel('RMSE')
plt.title('SVD RMSE vs Number of Latent Factors')
plt.tight_layout()
plt.savefig('task5_svd_k.png', dpi=100)
plt.show()

for kk, r in zip(k_list, svd_rmse):
    print(f'k={kk:3d}  RMSE={r:.4f}')


### Task 6: Surprise-Style SVD (SGD Matrix Factorization)

The `scikit-surprise` library is not available so Funk SVD is implemented from scratch using SGD.

In [ ]:
class SurpriseSVD:
    """
    Funk SVD (same formulation as scikit-surprise):
        r_hat = global_mean + b_u + b_i + p_u . q_i
    Trained with SGD on observed ratings only.
    """
    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02, seed=42):
        self.n_factors = n_factors
        self.n_epochs  = n_epochs
        self.lr        = lr
        self.reg       = reg
        self.seed      = seed

    def fit(self, train_df):
        np.random.seed(self.seed)
        self.mu = train_df['rating'].mean()

        self.user_enc = {u: i for i, u in enumerate(train_df['userId'].unique())}
        self.item_enc = {m: i for i, m in enumerate(train_df['movieId'].unique())}

        n_u = len(self.user_enc)
        n_i = len(self.item_enc)

        self.P  = np.random.normal(0, 0.1, (n_u, self.n_factors))
        self.Q  = np.random.normal(0, 0.1, (n_i, self.n_factors))
        self.bu = np.zeros(n_u)
        self.bi = np.zeros(n_i)

        data = train_df[['userId', 'movieId', 'rating']].values

        for epoch in range(self.n_epochs):
            np.random.shuffle(data)
            loss = 0
            for u_raw, i_raw, r in data:
                u = self.user_enc.get(int(u_raw))
                i = self.item_enc.get(int(i_raw))
                if u is None or i is None:
                    continue

                err   = r - (self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i])
                loss += err ** 2

                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (err - self.reg * self.bi[i])

                p_u = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * p_u       - self.reg * self.Q[i])

            if (epoch + 1) % 5 == 0:
                print(f'  epoch {epoch+1}/{self.n_epochs}  loss: {loss / len(data):.4f}')

        return self

    def predict(self, user_id, movie_id):
        u = self.user_enc.get(user_id)
        i = self.item_enc.get(movie_id)
        if u is None or i is None:
            return self.mu
        return float(np.clip(self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i], 0.5, 5.0))


In [ ]:
print('training Surprise-style SVD...')
surprise_svd = SurpriseSVD(n_factors=50, n_epochs=20, lr=0.005, reg=0.02)
surprise_svd.fit(train_ratings)


In [ ]:
a_surp, p_surp = [], []
for _, row in test_sample.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    a_surp.append(row['rating'])
    p_surp.append(surprise_svd.predict(uid, mid))

rmse_surprise = np.sqrt(mean_squared_error(a_surp, p_surp))
print(f'Surprise SVD RMSE : {rmse_surprise:.4f}')
print(f'NumPy SVD RMSE    : {rmse_svd:.4f}')
print()
print('Surprise SVD is better because it trains only on observed ratings,')
print('avoiding the noise that comes from filling missing values.')

def surprise_precision_recall(user_id, svd_model, test_ratings, train_ratings,
                               k=10, threshold=4.0):
    gt = test_ratings[
        (test_ratings['userId'] == user_id) & (test_ratings['rating'] >= threshold)
    ]['movieId'].tolist()
    if not gt:
        return None, None

    rated   = train_ratings[train_ratings['userId'] == user_id]['movieId'].tolist()
    all_ids = list(svd_model.item_enc.keys())
    unrated = [m for m in all_ids if m not in rated]

    preds   = {m: svd_model.predict(user_id, m) for m in unrated[:500]}
    top_k   = sorted(preds, key=preds.get, reverse=True)[:k]
    hits    = len(set(top_k) & set(gt))
    return hits / k, hits / len(gt)

p_sv2, r_sv2 = [], []
for uid in test_ratings['userId'].unique()[:50]:
    p, r = surprise_precision_recall(uid, surprise_svd, test_ratings, train_ratings)
    if p is not None:
        p_sv2.append(p); r_sv2.append(r)

print(f'\nSurprise SVD  Precision@10: {np.mean(p_sv2):.4f}  Recall@10: {np.mean(r_sv2):.4f}')


---
## Part 4 - Hybrid Recommendation Model

### Task 7: Meta-Learning Hybrid

Train a Ridge Regression meta-model that blends CBF and CF scores dynamically.

In [ ]:
movie_avg_rating = ratings.groupby('movieId')['rating'].mean().to_dict()
user_avg_rating  = ratings.groupby('userId')['rating'].mean().to_dict()

def get_cbf_score(user_id, movie_id, user_profiles, tfidf_matrix, movie_idx_map):
    if user_id not in user_profiles or movie_id not in movie_idx_map.index:
        return 0.0
    profile = user_profiles[user_id].reshape(1, -1)
    m_idx   = movie_idx_map[movie_id]
    return float(cosine_similarity(profile, tfidf_matrix[m_idx])[0, 0])

print('building meta-model features...')
meta_sample = train_ratings.sample(min(3000, len(train_ratings)), random_state=42)

X_meta, y_meta = [], []
for _, row in meta_sample.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    cbf  = get_cbf_score(uid, mid, user_profiles, tfidf_matrix, movie_idx_map)
    cf   = surprise_svd.predict(uid, mid)
    mavg = movie_avg_rating.get(mid, 3.5)
    uavg = user_avg_rating.get(uid, 3.5)
    X_meta.append([cbf, cf, mavg, uavg])
    y_meta.append(row['rating'])

X_meta = np.array(X_meta)
y_meta = np.array(y_meta)
print(f'feature matrix: {X_meta.shape}')


In [ ]:
X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(X_meta, y_meta, test_size=0.2, random_state=42)

meta_model = Ridge(alpha=1.0)
meta_model.fit(X_m_tr, y_m_tr)

y_m_pred   = np.clip(meta_model.predict(X_m_te), 0.5, 5.0)
rmse_hybrid_val = np.sqrt(mean_squared_error(y_m_te, y_m_pred))

print(f'Hybrid validation RMSE: {rmse_hybrid_val:.4f}')
print(f'Coefficients -> CBF: {meta_model.coef_[0]:.4f}  CF: {meta_model.coef_[1]:.4f}  '
      f'movie_avg: {meta_model.coef_[2]:.4f}  user_avg: {meta_model.coef_[3]:.4f}')


In [ ]:
test_hybrid = test_ratings.sample(min(500, len(test_ratings)), random_state=1)
a_h, p_h = [], []

for _, row in test_hybrid.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    cbf  = get_cbf_score(uid, mid, user_profiles, tfidf_matrix, movie_idx_map)
    cf   = surprise_svd.predict(uid, mid)
    mavg = movie_avg_rating.get(mid, 3.5)
    uavg = user_avg_rating.get(uid, 3.5)
    p    = np.clip(meta_model.predict([[cbf, cf, mavg, uavg]])[0], 0.5, 5.0)
    a_h.append(row['rating'])
    p_h.append(p)

rmse_hybrid_test = np.sqrt(mean_squared_error(a_h, p_h))
print(f'Hybrid test RMSE      : {rmse_hybrid_test:.4f}')
print(f'Surprise SVD test RMSE: {rmse_surprise:.4f}')

methods = ['User-CF', 'Item-CF', 'SVD', 'Surprise SVD', 'Hybrid']
rmses   = [rmse_ucf, rmse_icf, rmse_svd, rmse_surprise, rmse_hybrid_test]
colors  = ['steelblue', 'coral', 'green', 'purple', 'gold']

plt.figure(figsize=(9, 5))
bars = plt.bar(methods, rmses, color=colors)
for bar, val in zip(bars, rmses):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', fontsize=10)
plt.ylabel('RMSE')
plt.title('RMSE Comparison - All Methods')
plt.tight_layout()
plt.savefig('task7_rmse_all.png', dpi=100)
plt.show()


In [ ]:
user_counts  = train_ratings.groupby('userId').size()
cold_users   = user_counts[user_counts <= 5].index.tolist()
warm_users   = user_counts[user_counts > 20].index.tolist()

print(f'cold-start users (<=5 ratings) : {len(cold_users)}')
print(f'warm users (>20 ratings)       : {len(warm_users)}')

def hybrid_rmse_for_group(user_ids, test_ratings, svd_model, meta_model):
    a_list, p_list = [], []
    subset = test_ratings[test_ratings['userId'].isin(user_ids)]
    for _, row in subset.iterrows():
        uid, mid = int(row['userId']), int(row['movieId'])
        cbf  = get_cbf_score(uid, mid, user_profiles, tfidf_matrix, movie_idx_map)
        cf   = svd_model.predict(uid, mid)
        mavg = movie_avg_rating.get(mid, 3.5)
        uavg = user_avg_rating.get(uid, 3.5)
        p    = np.clip(meta_model.predict([[cbf, cf, mavg, uavg]])[0], 0.5, 5.0)
        a_list.append(row['rating'])
        p_list.append(p)
    return np.sqrt(mean_squared_error(a_list, p_list)) if a_list else np.nan

rmse_cold = hybrid_rmse_for_group(cold_users, test_ratings, surprise_svd, meta_model)
rmse_warm = hybrid_rmse_for_group(warm_users, test_ratings, surprise_svd, meta_model)

print(f'Hybrid RMSE - cold-start: {rmse_cold:.4f}')
print(f'Hybrid RMSE - warm users: {rmse_warm:.4f}')
print()
print('Cold-start performance is worse but the CBF component helps by using')
print('genre preferences instead of relying solely on rating history.')


---
## Part 5 - Learning-Based Recommender Systems

### Task 8: Neural Network Content-Based Filtering

In [ ]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['year'].fillna(movies['year'].median(), inplace=True)

all_genres = sorted(set(
    g for gs in movies['genres'].str.split('|') for g in gs
    if g != '(no genres listed)'
))
print(f'{len(all_genres)} genres:', all_genres)

for g in all_genres:
    movies[g] = movies['genres'].str.contains(g, regex=False).astype(int)

movie_avg = ratings.groupby('movieId')['rating'].mean().reset_index()
movie_avg.columns = ['movieId', 'avg_rating']
movies = movies.merge(movie_avg, on='movieId', how='left')
movies['avg_rating'].fillna(movies['avg_rating'].median(), inplace=True)

movie_feature_cols = all_genres + ['year', 'avg_rating']
print(f'movie feature columns: {len(movie_feature_cols)}')


In [ ]:
# user features: average rating per genre
ratings_with_genres = ratings.merge(movies[['movieId'] + all_genres], on='movieId', how='left')

user_genre_avg = {}
for uid, group in ratings_with_genres.groupby('userId'):
    row = {}
    for g in all_genres:
        genre_ratings = group[group[g] == 1]['rating']
        row[f'user_avg_{g}'] = genre_ratings.mean() if len(genre_ratings) > 0 else 3.5
    user_genre_avg[uid] = row

user_feat_df   = pd.DataFrame.from_dict(user_genre_avg, orient='index').fillna(3.5)
user_feat_cols = user_feat_df.columns.tolist()
print(f'user feature columns: {len(user_feat_cols)}')
user_feat_df.head(3)


In [ ]:
movie_feat_lookup = movies.set_index('movieId')[movie_feature_cols]

X_user_list, X_movie_list, y_nn = [], [], []

for _, row in train_ratings.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if uid not in user_feat_df.index or mid not in movie_feat_lookup.index:
        continue
    X_user_list.append(user_feat_df.loc[uid].values.astype(float))
    X_movie_list.append(movie_feat_lookup.loc[mid].values.astype(float))
    y_nn.append(row['rating'])

X_combined = np.hstack([np.array(X_user_list), np.array(X_movie_list)])
y_nn       = np.array(y_nn)

print(f'training samples  : {X_combined.shape[0]}')
print(f'user feature dim  : {np.array(X_user_list).shape[1]}')
print(f'movie feature dim : {np.array(X_movie_list).shape[1]}')
print(f'combined input dim: {X_combined.shape[1]}')


In [ ]:
scaler = StandardScaler()
X_tr_nn, X_val_nn, y_tr_nn, y_val_nn = train_test_split(X_combined, y_nn,
                                                          test_size=0.15, random_state=42)
X_tr_sc  = scaler.fit_transform(X_tr_nn)
X_val_sc = scaler.transform(X_val_nn)

# two-branch MLP flattened into a single hidden stack after concatenation
nn_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
    verbose=False
)

print('training neural network...')
nn_model.fit(X_tr_sc, y_tr_nn)

y_val_pred = np.clip(nn_model.predict(X_val_sc), 0.5, 5.0)
rmse_nn_val = np.sqrt(mean_squared_error(y_val_nn, y_val_pred))
print(f'validation RMSE: {rmse_nn_val:.4f}')

plt.figure()
plt.plot(nn_model.loss_curve_, color='steelblue', label='train loss')
plt.xlabel('epoch')
plt.ylabel('MSE')
plt.title('Neural Network Training Loss')
plt.legend()
plt.tight_layout()
plt.savefig('task8_nn_loss.png', dpi=100)
plt.show()


In [ ]:
X_test_list, y_test_list = [], []
for _, row in test_ratings.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if uid not in user_feat_df.index or mid not in movie_feat_lookup.index:
        continue
    X_test_list.append(np.hstack([
        user_feat_df.loc[uid].values.astype(float),
        movie_feat_lookup.loc[mid].values.astype(float)
    ]))
    y_test_list.append(row['rating'])

X_test_nn = np.array(X_test_list)
y_test_nn = np.array(y_test_list)
X_test_sc = scaler.transform(X_test_nn)

y_test_pred_nn = np.clip(nn_model.predict(X_test_sc), 0.5, 5.0)
rmse_nn_test   = np.sqrt(mean_squared_error(y_test_nn, y_test_pred_nn))

print(f'NN test RMSE          : {rmse_nn_test:.4f}')
print(f'Surprise SVD test RMSE: {rmse_surprise:.4f}')
print(f'Hybrid test RMSE      : {rmse_hybrid_test:.4f}')
print()
print('The NN captures non-linear interactions between genre preferences and movie')
print('attributes that TF-IDF cosine similarity misses. It underperforms CF methods')
print('here mainly because the dataset is small and sparse.')


### Task 9: Reinforcement Learning in Recommender Systems

In [ ]:
# offline reward signal: rating >= 4 is positive, below 4 is negative
reward_lookup = {}
for _, row in ratings.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    reward_lookup[(uid, mid)] = 1 if row['rating'] >= 4 else -1

movie_id_list = movies['movieId'].tolist()
n_movies      = len(movie_id_list)

print(f'users: {ratings["userId"].nunique()}  movies (arms): {n_movies}')
print(f'positive rewards: {sum(1 for v in reward_lookup.values() if v == 1)}')
print(f'negative rewards: {sum(1 for v in reward_lookup.values() if v == -1)}')


In [ ]:
class EpsilonGreedyBandit:
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms      = n_arms
        self.epsilon     = epsilon
        self.q           = np.zeros(n_arms)
        self.counts      = np.zeros(n_arms)
        self.n_explore   = 0
        self.n_exploit   = 0

    def select(self):
        if np.random.rand() < self.epsilon:
            self.n_explore += 1
            return np.random.randint(self.n_arms)
        self.n_exploit += 1
        return np.argmax(self.q)

    def update(self, arm, reward):
        self.counts[arm] += 1
        self.q[arm] += (reward - self.q[arm]) / self.counts[arm]


class UCBBandit:
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c      = c
        self.q      = np.zeros(n_arms)
        self.counts = np.zeros(n_arms)
        self.t      = 0

    def select(self):
        self.t += 1
        unseen = np.where(self.counts == 0)[0]
        if len(unseen) > 0:
            return unseen[0]
        ucb = self.q + self.c * np.sqrt(np.log(self.t) / self.counts)
        return np.argmax(ucb)

    def update(self, arm, reward):
        self.counts[arm] += 1
        self.q[arm] += (reward - self.q[arm]) / self.counts[arm]


print('bandit classes ready')


In [ ]:
def simulate_mab(bandit, reward_lookup, movie_id_list, user_ids, n_rounds):
    cumulative = []
    total = 0
    for _ in range(n_rounds):
        uid = np.random.choice(user_ids)
        arm = bandit.select()
        mid = movie_id_list[arm]
        r   = reward_lookup.get((uid, mid), 0)
        bandit.update(arm, r)
        total += r
        cumulative.append(total)
    return cumulative

sample_users_rl = ratings['userId'].unique()[:100].tolist()
N_ROUNDS = 1000

eps_bandit = EpsilonGreedyBandit(n_movies, epsilon=0.1)
ucb_bandit = UCBBandit(n_movies, c=2.0)

print(f'simulating {N_ROUNDS} rounds...')
rewards_eps = simulate_mab(eps_bandit, reward_lookup, movie_id_list, sample_users_rl, N_ROUNDS)
rewards_ucb = simulate_mab(ucb_bandit, reward_lookup, movie_id_list, sample_users_rl, N_ROUNDS)

print(f'epsilon-greedy: explore={eps_bandit.n_explore}  exploit={eps_bandit.n_exploit}')
print(f'epsilon-greedy cumulative reward: {rewards_eps[-1]}')
print(f'UCB cumulative reward           : {rewards_ucb[-1]}')

plt.figure()
plt.plot(rewards_eps, label='epsilon-greedy', color='steelblue')
plt.plot(rewards_ucb, label='UCB (c=2)',      color='orange')
plt.xlabel('round')
plt.ylabel('cumulative reward')
plt.title('MAB: epsilon-greedy vs UCB')
plt.legend()
plt.tight_layout()
plt.savefig('task9_mab.png', dpi=100)
plt.show()


In [ ]:
class QLearningAgent:
    """
    State: last reward context  (0=neutral, 1=positive, 2=negative)
    Action: movie index
    """
    def __init__(self, n_actions, n_states=3, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.n_actions = n_actions
        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = epsilon
        self.Q         = np.random.uniform(0, 0.01, (n_states, n_actions))

    def state_from_reward(self, r):
        return 1 if r > 0 else (2 if r < 0 else 0)

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])

    def update(self, s, a, r, s_next):
        td = r + self.gamma * np.max(self.Q[s_next]) - self.Q[s, a]
        self.Q[s, a] += self.alpha * td


# limit action space to top 500 movies to keep Q-table manageable
n_ql_arms     = 500
top_movie_ids = ratings.groupby('movieId').size().nlargest(n_ql_arms).index.tolist()

agent     = QLearningAgent(n_actions=n_ql_arms)
state     = 0
cum_ql    = []
total_ql  = 0

for _ in range(N_ROUNDS):
    uid    = np.random.choice(sample_users_rl)
    action = agent.act(state)
    mid    = top_movie_ids[action]
    reward = reward_lookup.get((uid, mid), 0)
    
    s_next = agent.state_from_reward(reward)
    agent.update(state, action, reward, s_next)
    state  = s_next

    total_ql += reward
    cum_ql.append(total_ql)

print(f'Q-learning cumulative reward: {cum_ql[-1]}')

plt.figure()
plt.plot(rewards_eps, label='epsilon-greedy', color='steelblue')
plt.plot(rewards_ucb, label='UCB',            color='orange')
plt.plot(cum_ql,      label='Q-learning',     color='green')
plt.xlabel('round')
plt.ylabel('cumulative reward')
plt.title('RL Agents Comparison')
plt.legend()
plt.tight_layout()
plt.savefig('task9_rl.png', dpi=100)
plt.show()


In [ ]:
top_arms    = np.argsort(eps_bandit.q)[::-1][:10]
top_eps_ids = [movie_id_list[a] for a in top_arms]
rl_recs     = movies[movies['movieId'].isin(top_eps_ids)][['title', 'genres']].copy()
rl_recs['Est. Reward'] = [eps_bandit.q[a] for a in top_arms]

print('Epsilon-greedy top 10 recommendations (global):')
print(rl_recs.to_string(index=False))

explore_pct = eps_bandit.n_explore / (eps_bandit.n_explore + eps_bandit.n_exploit)
print(f'\nexploration rate: {explore_pct:.2%}')
print()
print('RL vs traditional methods:')
print('  epsilon-greedy: simple, fast, fixed 10% exploration, no personalisation')
print('  UCB           : smarter exploration based on uncertainty, still global')
print('  Q-learning    : state-aware, learns from sequential feedback per context')
print('  SVD/CF        : offline, accurate on dense data, no exploration at all')


---
## Part 6 - Explainability

### Task 10: Feature-Based Explanations (Content-Based Filtering)

In [ ]:
def permutation_importance(model, scaler, X_sample, y_sample, feature_names, n_repeats=5):
    X_sc     = scaler.transform(X_sample)
    base_mse = mean_squared_error(y_sample, np.clip(model.predict(X_sc), 0.5, 5.0))
    result   = {}
    for i, name in enumerate(feature_names):
        scores = []
        for _ in range(n_repeats):
            X_perm = X_sc.copy()
            X_perm[:, i] = np.random.permutation(X_perm[:, i])
            mse = mean_squared_error(y_sample, np.clip(model.predict(X_perm), 0.5, 5.0))
            scores.append(mse)
        result[name] = np.mean(scores) - base_mse
    return result

sample_idx = np.random.choice(len(X_test_nn), size=min(200, len(X_test_nn)), replace=False)
X_expl     = X_test_nn[sample_idx]
y_expl     = y_test_nn[sample_idx]

feature_names_all = (
    [f'user_{g}' for g in all_genres] +
    [f'movie_{g}' for g in all_genres] +
    ['movie_year', 'movie_avg_rating']
)

print('computing permutation importance...')
imp = permutation_importance(nn_model, scaler, X_expl, y_expl, feature_names_all)

imp_series = pd.Series(imp).sort_values(ascending=False)
top15      = imp_series.head(15)

plt.figure(figsize=(10, 6))
bar_colors = ['tomato' if v > 0 else 'steelblue' for v in top15.values]
plt.barh(top15.index[::-1], top15.values[::-1], color=bar_colors[::-1])
plt.xlabel('increase in MSE when feature shuffled')
plt.title('Feature Importance (permutation / SHAP approximation)')
plt.tight_layout()
plt.savefig('task10_importance.png', dpi=100)
plt.show()
print(top15.to_string())


In [ ]:
def explain_cbf(user_id, movie_title, user_feat_df, movies, all_genres):
    movie_row = movies[movies['title'] == movie_title]
    if movie_row.empty or user_id not in user_feat_df.index:
        return 'not found'

    movie_genres    = [g for g in all_genres if movie_row.iloc[0][g] == 1]
    user_scores     = user_feat_df.loc[user_id][[f'user_{g}' for g in all_genres]]
    top_user_genres = user_scores.nlargest(3).index.str.replace('user_', '').tolist()
    overlap         = [g for g in movie_genres if g in top_user_genres]
    year            = int(movie_row.iloc[0]['year'])

    lines = [
        f"Explanation for user {user_id} -> '{movie_title}'",
        f"  movie genres      : {', '.join(movie_genres)}",
        f"  your top genres   : {', '.join(top_user_genres)}",
        f"  overlap           : {', '.join(overlap) if overlap else 'none'}",
        f"  release year      : {year}",
        f"  avg rating        : {movie_row.iloc[0]['avg_rating']:.2f}",
        f"  -> recommended because you like {', '.join(overlap or movie_genres[:2])} films",
    ]
    return '\n'.join(lines)

print(explain_cbf(1, 'Toy Story (1995)', user_feat_df, movies, all_genres))
print()
print(explain_cbf(5, 'Goodfellas (1990)', user_feat_df, movies, all_genres))


### Task 11: Neighborhood-Based Explanations (Collaborative Filtering)

In [ ]:
def explain_user_cf(user_id, movie_id, sim_df, matrix, movies_df, k=5):
    if user_id not in sim_df.index:
        return 'user not found'
    movie_row = movies_df[movies_df['movieId'] == movie_id]
    if movie_row.empty:
        return 'movie not found'

    title   = movie_row.iloc[0]['title']
    raters  = matrix[movie_id].dropna().index.difference([user_id])
    if raters.empty:
        return 'no neighbors rated this movie'

    sims = sim_df.loc[user_id, raters].nlargest(k)
    lines = [f"User-CF explanation: user {user_id} -> '{title}'",
             f"  {'neighbor':>10}  {'similarity':>10}  {'their rating':>12}"]
    for neighbor, sim in sims.items():
        rating = matrix.loc[neighbor, movie_id]
        lines.append(f"  {neighbor:>10}  {sim:>10.4f}  {rating:>12.1f}")

    avg_r = (sims * matrix.loc[sims.index, movie_id]).sum() / sims.abs().sum()
    lines.append(f"  -> neighbors similar to you rated this film ~{avg_r:.2f}/5")
    return '\n'.join(lines)

print(explain_user_cf(1, 318, user_sim_df, user_movie_matrix, movies))


In [ ]:
def explain_item_cf(user_id, target_movie_id, item_sim, item_id_to_idx, matrix, movies_df, k=5):
    movie_row = movies_df[movies_df['movieId'] == target_movie_id]
    if movie_row.empty:
        return 'movie not found'
    target_title = movie_row.iloc[0]['title']
    target_idx   = item_id_to_idx.get(target_movie_id)
    if target_idx is None:
        return 'movie not in similarity matrix'

    rated       = matrix.loc[user_id].dropna()
    rated_ids   = [m for m in rated.index if m in item_id_to_idx]
    rated_idxs  = [item_id_to_idx[m] for m in rated_ids]

    sims    = item_sim[target_idx, rated_idxs]
    top_k   = np.argsort(np.abs(sims))[::-1][:k]

    lines = [f"Item-CF explanation: why '{target_title}' for user {user_id}",
             "  similar movies you have already rated:"]
    for i in top_k:
        mid   = rated_ids[i]
        row   = movies_df[movies_df['movieId'] == mid]
        t     = row.iloc[0]['title'][:40] if not row.empty else str(mid)
        ur    = rated.loc[mid]
        lines.append(f"    '{t}'  your rating: {ur:.1f}  similarity: {sims[i]:.4f}")

    lines.append(f"  -> because you liked these, '{target_title}' is a likely match")
    return '\n'.join(lines)

print(explain_item_cf(1, 318, item_sim, item_id_to_idx, user_movie_matrix, movies))


### Task 12: Model-Agnostic Explainability (LIME for Neural Network)

In [ ]:
def lime_explain(model, scaler, instance, feature_names, n_samples=200, sigma=0.1):
    instance_sc = scaler.transform(instance.reshape(1, -1))[0]

    noise     = np.random.normal(0, sigma, (n_samples, len(instance_sc)))
    perturbed = instance_sc + noise

    preds     = np.clip(model.predict(perturbed), 0.5, 5.0)
    distances = np.sqrt(((perturbed - instance_sc) ** 2).sum(axis=1))
    weights   = np.exp(-(distances ** 2) / (2 * sigma ** 2))

    local_model = Ridge(alpha=1.0)
    local_model.fit(perturbed, preds, sample_weight=weights)

    return dict(zip(feature_names, local_model.coef_))

sample_inst = X_test_nn[0]
true_rating = y_test_nn[0]
pred_rating = np.clip(nn_model.predict(scaler.transform(sample_inst.reshape(1,-1))), 0.5, 5.0)[0]

print(f'true rating: {true_rating:.1f}  predicted: {pred_rating:.3f}')

lime_imp   = lime_explain(nn_model, scaler, sample_inst, feature_names_all)
lime_series = pd.Series(lime_imp).sort_values(key=abs, ascending=False).head(15)

plt.figure(figsize=(10, 6))
bar_c = ['tomato' if v > 0 else 'steelblue' for v in lime_series.values]
plt.barh(lime_series.index[::-1], lime_series.values[::-1], color=bar_c[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('LIME coefficient (positive = pushes prediction higher)')
plt.title('LIME Local Explanation - Neural Network')
plt.tight_layout()
plt.savefig('task12_lime.png', dpi=100)
plt.show()
print(lime_series.to_string())


### Task 13: Evaluating Explainability

In [ ]:
print("""
Explainability Evaluation
--------------------------

1. Feature-based (permutation importance / SHAP):
   Makes recommendations clearer by showing which genres and metadata
   drove the prediction. Also good for auditing bias - if avg_rating
   (popularity) dominates then the model is favoring popular titles
   regardless of personal taste.

2. Neighborhood-based (k-NN, Task 11):
   Most intuitive for users. Saying 'people similar to you rated this
   5/5' is immediately actionable. Can reveal echo-chamber bias when
   the neighbor cluster is homogeneous.

3. LIME (Task 12):
   Best for debugging individual surprising predictions. Shows which
   features push the score up or down locally. Can catch interactions
   that global importance misses, e.g. a user who likes Action+Drama
   combos but not pure Action or pure Drama alone.

Overall: neighborhood explanations are easiest to understand, SHAP is
best for system-level auditing, LIME is best for per-prediction debugging.
""")


In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'  {"method":<22}  {"RMSE":>8}')
print('-' * 40)
print(f'  {"User-CF (K=20)":<22}  {rmse_ucf:>8.4f}')
print(f'  {"Item-CF (K=20)":<22}  {rmse_icf:>8.4f}')
print(f'  {"SVD numpy (k=50)":<22}  {rmse_svd:>8.4f}')
print(f'  {"Surprise SVD (SGD)":<22}  {rmse_surprise:>8.4f}')
print(f'  {"Neural Network":<22}  {rmse_nn_test:>8.4f}')
print(f'  {"Hybrid":<22}  {rmse_hybrid_test:>8.4f}')
print('=' * 60)
